In [40]:
import openai
import json
from tqdm import tqdm
import pandas as pd
import time

In [2]:
# Set up OpenAI API key
openai.api_key = 'XXX'

In [49]:
def extract_info_from_project_data(project_data):
    prompt = f"""
    Given the following project data in JSON format, extract the following information:
    1. Organisms used in searches (combine name and accession, separated by a semicolon | for multiple entries, no space in between). Use the same format for all fields, if multiple entries.
    2. Type of acquisition (DDA, DIA, crosslinking or XL-MS, or other). Use these values, and use your judgement if not explicitly stated. For example, if the description mentions "data-independent acquisition", you can assume it is DIA. Values like HdX or hydrogen-deuterium exchange can also have their own category.
    3. Modifications used in the searches (combine name and accession, separated by a semicolon | for multiple entries, no spaces).
    4. If it looks like a large scale study or not (two values, True or False) based on the description. Large scale study would be defined as a study that involves a large number of samples and raw files, more than 50 as a rule of thumb.
    5. Instrument name. Should be available in the "instruments" array in the JSON data, otherwise use your judgement from the description.
    6. Detector type (examples include time of flight, orbitrap, ion trap). Use your judgement if not explicitly stated.
    7. Fragmentation method if possible. Again, use your judgement if not explicitly stated. If it is not possible to determine, use "N/A".
    8. Tissue or organism parts used in the study. If not explicitly stated, use "N/A". Use "full" if the whole organism was used.
    9. Disease or condition studied. If not explicitly stated, use "N/A".
    10. Cell type or cell line used. If not explicitly stated, use "N/A".
    11. Experiment type (examples include shotgun proteomics, targeted proteomics, metabolomics, etc.). Use your judgement if not explicitly stated.
    12. Quantification method used. If not explicitly stated, use "N/A".

    Especially on modifications, instruments, and fragmentation methods, use your judgement and knowledge of the field to make the best guess if the information is not explicitly stated. If you are not sure, you can leave it blank.
    For modifications, some of the entries are synonyms, such as "oxidation" and "monohydroxylated residue", or "carbamidomethylation" and "iodoacetamide derivatized residue". Use your knowledge of proteomics and protein modifications along with your judgement to determine if they are the same modification or not. 
    Some of the studies use labeling techniques, such as SILAC or TMT. These should be included in the modifications field, as they are chemical modifications of the peptides. No need to list the specific amino acid residues that are modified, just the general modification type. For example, TMT11plex, TMT6plex, TMTpro, iTRAQ would be sufficient. 

    The extracted information should be in the following format:
    {{
     "orgs": "Homo sapiens (human);9606",
     "acquisition": "DDA",
     "mods": "monohydroxylated residue;MOD:00425",
     "large_study": "True",
     "instrument_name": "Q Exactive",
     "detector_type": "Orbitrap",
     "fragmentation_method": "Higher-energy collisional dissociation (HCD)"
     "tissue": "Heart"
     "disease": "Cardiovascular"
     "cell_type": "neutrophils"
     "experiment_type": "shotgun proteomics"
     "quant": "label free"
     }}

    Do not output anything else other than the object requested. No explanation or discussion is needed. Now, process the following project data:

    {json.dumps(project_data, indent=4)}
    """

    response = openai.ChatCompletion.create(
        model="gpt-4o",  # or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        n=1,
        stop=None,
        temperature=0
    )

    return response['choices'][0]['message']['content'].strip()

In [7]:
# read data
# read data, each line in the txt is a json object
data = []
with open('all_projects_retrieve.txt', 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(data[0])

{'accession': 'PXD049090', 'title': 'The HisRS-like domain of GCN2 is a pseudoenzyme that can bind uncharged tRNA - HX-MS Data', 'additionalAttributes': [{'@type': 'CvParam', 'cvLabel': 'PRIDE', 'accession': 'PRIDE:0000411', 'name': 'Dataset FTP location', 'value': 'ftp://ftp.pride.ebi.ac.uk/pride/data/archive/2024/02/PXD049090'}], 'projectDescription': 'GCN2 is a stress response kinase that phosphorylates the translation initiation factor eIF2\uf061\uf020to inhibit general protein synthesis when activated by uncharged tRNA and stalled ribosomes. The presence of a HisRS-like domain in GCN2, normally associated with the ability to bind and aminoacylate tRNAs, led to the hypothesis that eIF2\uf061 kinase activity is regulated by the direct binding of this domain to uncharged tRNA. Here we solved the structure of the HisRS-like domain in the context of full-length GCN2 by cryoEM. Structure and function analysis shows the HisRS-like domain of GCN2 has lost tRNA charging, ATP binding, and h

In [51]:
parsed_accessions = []
with open('parsed_accessions.txt', 'r') as f:
    for line in f:
        parsed_accessions.append(line.strip())
        
with open('parsed_accessions.txt', 'a') as accessions_file:
    with open('parse_data.tsv', 'a') as f:        
        for project in tqdm(data[:1000]):
            accession = project["accession"]

            if accession not in parsed_accessions:
                date = project["submissionDate"]
                title = project["title"]

                project_info = extract_info_from_project_data(project)
                print(project_info)
                project_info = project_info.replace("```", "").replace("json", "").strip()
                project_info = json.loads(project_info)

                # add accession, date, title to the project_info
                project_info["accession"] = accession
                project_info["date"] = date
                project_info["title"] = title

                # write to file in tsv format
                f.write("\t".join([str(project_info[key]) for key in project_info.keys()]) + "\n")
                accessions_file.write(accession + "\n")

                # sleep for 4 seconds
                time.sleep(4)


100%|██████████| 1000/1000 [00:00<00:00, 79561.14it/s]


In [52]:
df = pd.read_csv('parse_data.tsv', header=None, sep='\t')

In [53]:
df.columns = ["orgs", "acquisition", "mods", "large_study", "instrument_name", "detector_type", "fragmentation_method", "tissue", "disease", "cell_type", "experiment_type", "quant", "accession", "date", "title"]
df.sample(5)

,orgs,acquisition,mods,large_study,instrument_name,detector_type,fragmentation_method,tissue,disease,cell_type,experiment_type,quant,accession,date,title
677,Coniochaeta ligniaria nrrl 30616;1408157,DDA,monohydroxylated residue;MOD:00425|deamidated ...,False,LTQ Orbitrap Elite,Orbitrap,Higher-energy collisional dissociation (HCD),NaN,NaN,NaN,shotgun proteomics,TMT,PXD045361,2023-09-13,Coniochaeta ligniaria NRRL 30616 v1.0 LC-MSMS
318,Arabidopsis thaliana (mouse-ear cress);3702,DDA,phosphorylated residue;MOD:00696|acetylated re...,False,Orbitrap Fusion Lumos,Orbitrap,NaN,Plant cell,NaN,NaN,shotgun proteomics,label free,PXD046788,2023-11-08,Unraveling epigenetic changes in A. thaliana c...
876,Solanum lycopersicum;4081,DDA,acetylated residue;MOD:00394|iodoacetamide der...,False,impact II,time of flight,NaN,Leaf,NaN,NaN,shotgun proteomics,Dimethyl Labeling,PXD044637,2023-08-18,N-terminome analysis of tomato P69D knock-out ...
139,Homo sapiens (human);9606,DDA,carbamidomethylation;MOD:00487|oxidation;MOD:0...,False,Q Exactive HF,Orbitrap,Higher-energy collisional dissociation (HCD),NaN,Brain glioblastoma multiforme|Brain cancer,U87MG glioblastoma cells,shotgun proteomics,label free,PXD047816,2023-12-14,Proteomic analysis of purified extracellular v...
7,Mus musculus (mouse);10090,DIA,monohydroxylated residue;MOD:00425|iodoacetami...,False,Orbitrap Exploris 480,Orbitrap,Higher-energy collisional dissociation (HCD),Primary cell,Leukemia,NaN,shotgun proteomics,NaN,PXD049014,2024-01-30,Quantitative secretome analysis of fetal and a...


In [61]:
df.describe()

,orgs,acquisition,mods,large_study,instrument_name,detector_type,fragmentation_method,tissue,disease,cell_type,experiment_type,quant,accession,date,title
count,1019,1000,872,1000,1000,997,631,599,316,294,1000,641,1000,1000,1000
unique,228,16,385,2,103,17,32,219,166,142,54,46,1000,172,971
top,Homo sapiens (human);9606,DDA,monohydroxylated residue;MOD:00425|iodoacetami...,False,Orbitrap Fusion Lumos,Orbitrap,Higher-energy collisional dissociation (HCD),Cell culture,Disease free,Epithelial cell,shotgun proteomics,label free,PXD049090,2023-08-18,Ultra-fast label-free quantification and
freq,410,801,98,898,161,842,514,67,26,23,741,277,1,20,6


In [62]:
df.to_csv('parsed_gpt.tsv', sep='\t', index=False)